# Работа с сетью (IP-адреса)

**IP-адрес** - уникальный адрес сетевого устройства, который позволяет однозначно его идентифицировать и корректно выстраивать коммуникацию и маршрутизацию в сети.

IP-адреса могут записываться в формате **IPv4** (более привычный и широко распространенный, ЕГЭ базируется только на нем) и в более современном **IPv6** (появился, когда IPv4-адреса начали заканчиваться).

| | IPv4 | IPv6|
| --- | --- | --- |
| запись | 4 октета (по 8 бит) | 8 гексетов (по 16 бит) |
| пример | 123.123.123.123 | aaaa:1234:ffff:5678:9abc:def0:0000:1111 |
| объем | 32 бита | 128 бит |

In [3]:
# Как найти максимальное количество занимаемых адресов каждой из версией адресных пространств?
IPv4_count = ?
IPv6_count = ?
print('IPv4 = ', IPv4_count, ' IPv6 = ', IPv6_count)

SyntaxError: invalid syntax (2909400192.py, line 2)

### Почему по-прежнему используются IPv4 адреса, если под всевозможные сетевые устройства уже нет достаточного количества уникальных адресов? 

Проблема коллизий (когда у нескольких устройств может оказаться общий адрес) решается системой NAT (Network Address Translation - преобразование сетевых адресов). При переходе через маршрутизатор локальные IP-адреса "схлопываются" в "открытый" общедоступный IP-адрес маршрутизатора, а сами устройства отличаются друг от друга разными портами (наиболее популярная схема PAT), но не всегда проблема коллизий решается портами, так, для микросервисной архитектуры наиболее удобна схема со статическим NAT. 

### Так почему все сразу не перешли на IPv6?

Потому что так исторически сложилось, по сути - процесс переезда на новый формат IP-адресов очень болезненный и радикальный, особенно для старой инфраструктуры, которая давно и стабильно работает по старым протоколам.

## IPv4 

IPv4 записывается четырьмя октетами (числами по 8 бит), записанными в десятичной системе счисления и разделенных точками. Пример: `200.100.123.012`. Максимально возможное значение октета: **255** (почему?). 

IP-адреса могут подразделяться на публичные и приватные ("серые"), публичные адреса непосредственно используются для выхода в глобальную сеть, приватные же используются внутри подсетей. Публичные распределяются Интернет-провайдерами, гарантируется, что они будут уникальными во всем Интернете, локальные же уникальны только в рамках подсети, настраиваются самостоятельно, произвольным образом. 

Также IP-адреса бывают динамическими и статическими. Статические IP-адреса "приватизируются" определенными устройствами и не меняются при переподключении (обычно за доп. плату провайдеру).

### Дополнительные термины

 - **Loopback (петлевой интерфейс)** - виртуальный сетевой интерфейс для общения сетевого устройства "самим с собой" (например, для отладки или проверки соединения), Loopback-адреса имеют диапазон `127.0.0.0/8` (**localhost**)
 - **Broadcast (широковещательный адрес)** - специальный адрес в подсети для отправки сообщений одновременно всем сетевым устройствам, чаще всего `255.255.255.255`
 - **Multicast** - более продвинутая версия broadcast'а, отправка идет строго очерченному кругу устройств, которые подписались на канал рассылки (broadcast вещает вообще всем каждый раз)
 - **Основной шлюз (Default Gateway)** - IP-адрес роутера, куда обращаются устройства в локальной сети (но это не публичный адрес для выхода в Интернет)
 - **DNS-сервер (Domain Name System)** - сервис, помогающий сопоставлять привычные адреса веб-сайтов в IP-адреса, по которым к ним можно обратиться
 - **DHCP-сервер (Dynamic Host Configuration Protocol)** - сервис, который помогает сконфигурировать сетевые настройки устройства (автоматически назначить IP-адрес, параметры по типу маски, шлюзов, DNS-сервера)
 - **MAC-адрес** - неизменный уникальный идентификационный номер самого сетевого устройства, который "прошивается" в момент изготовления (по аналогии c IMEI у телефонов и VIN у автомобилей), по MAC-адресам устройства общаются в рамках локальной сети, но их нельзя использовать в глобальной сети из-за невозможности маршрутизации по ним, также MAC-адреса легко подменить программно

### Маска подсети

Маска - битовой фильтр, который позволяет разделить IP-адрес на адрес самой сети и адрес устройства внутри неё. как и сам IP-адрес, маска 32-битная, сначала идет часть с единицами, отвечающая за адрес сети, затем - только нули - часть адреса устройства в подсети. 

 - `255.255.255.0` - наиболее популярный формат, нужно перевести его в двоичный формат, чтобы понять, сколько битов выделяется под адрес самой сети (24 единицы)
 - `/24` - другой формат, приписываемый в конце IP-адреса, сразу понятно, что на адрес сети уходят первые 24 бита. 

## Пакет Python ipaddress

 - `p_network(address, strict=True)` — создает объект сети по IP-адресу и маске. Если передать вторым аргументом 0 (или strict=False), Python будет в нестрогом режиме и проигнорирует неверные биты хоста, автоматически выставив их в ноль. Если strict=True, будут выдаваться ошибки
 - `ip_address(str)` — создает объект конкретного IP-адреса.
 - `net.network_address` — возвращает адрес сети.
 - `net.netmask` — возвращает маску сети.
 - `net.hosts()` — возвращает итератор по всем узлам сети (без адреса сети и широковещательного адреса).
 - `Оператор in` — проверяет принадлежность IP-адреса данной сети (ip in net)



In [4]:

# 1. Поиск адреса сети по узлу и маске
#Суть задачи: Дан IP-адрес конкретного компьютера (узла) и маска сети. Нужно найти адрес самой сети (или количество единиц/нулей в его двоичной записи).
#Параметр strict=False обязателен, иначе Python выдаст ошибку, так как мы передаем адрес узла, а не сети.

from ipaddress import ip_network

# Задаем узел и маску (можно в формате '192.168.1.50/255.255.255.0' или с префиксом '/24')
node_address = "114.120.185.150"
mask = "255.255.240.0"

# Создаем объект сети. strict=False позволяет передавать адрес узла вместо адреса сети
net = ip_network(f"{node_address}/{mask}", strict=False)

# Выводим адрес сети
print("Адрес сети:", net.network_address)

# Если нужно перевести в двоичный вид (например, посчитать количество единиц):
binary_net = "".join(f"{int(octet):08b}" for octet in str(net.network_address).split("."))
print("Сеть в двоичном виде:", binary_net)
print("Количество единиц в адресе сети:", binary_net.count("1"))

Адрес сети: 114.120.176.0
Сеть в двоичном виде: 01110010011110001011000000000000
Количество единиц в адресе сети: 11


In [5]:

# 2. Перебор масок для поиска подходящей сети

#Суть задачи: Даны два узла (или узел и адрес сети). Известно, что они находятся в одной подсети. Нужно найти возможную маску (или количество таких масок).
#Мы перебираем длину маски от 0 до 32 (количество единиц слева направо).

from ipaddress import ip_network

ip1 = "214.123.14.125"
ip2 = "214.123.14.150"

# Перебираем все возможные длины масок (от /0 до /32)
for mask_len in range(33):
    # Создаем сеть для первого IP с текущей маской
    net = ip_network(f"{ip1}/{mask_len}", strict=False)
    
    # Проверяем, входит ли второй IP в эту же сеть
    # (Метод ip_network с strict=False автоматически определит адрес сети для ip1)
    if ip_network(f"{ip2}/{mask_len}", strict=False) == net:
        print(f"Маска /{mask_len} подходит. Сама маска: {net.netmask}")

Маска /0 подходит. Сама маска: 0.0.0.0
Маска /1 подходит. Сама маска: 128.0.0.0
Маска /2 подходит. Сама маска: 192.0.0.0
Маска /3 подходит. Сама маска: 224.0.0.0
Маска /4 подходит. Сама маска: 240.0.0.0
Маска /5 подходит. Сама маска: 248.0.0.0
Маска /6 подходит. Сама маска: 252.0.0.0
Маска /7 подходит. Сама маска: 254.0.0.0
Маска /8 подходит. Сама маска: 255.0.0.0
Маска /9 подходит. Сама маска: 255.128.0.0
Маска /10 подходит. Сама маска: 255.192.0.0
Маска /11 подходит. Сама маска: 255.224.0.0
Маска /12 подходит. Сама маска: 255.240.0.0
Маска /13 подходит. Сама маска: 255.248.0.0
Маска /14 подходит. Сама маска: 255.252.0.0
Маска /15 подходит. Сама маска: 255.254.0.0
Маска /16 подходит. Сама маска: 255.255.0.0
Маска /17 подходит. Сама маска: 255.255.128.0
Маска /18 подходит. Сама маска: 255.255.192.0
Маска /19 подходит. Сама маска: 255.255.224.0
Маска /20 подходит. Сама маска: 255.255.240.0
Маска /21 подходит. Сама маска: 255.255.248.0
Маска /22 подходит. Сама маска: 255.255.252.0
Маска 

In [6]:

# 3. Подсчет и перебор всех узлов в сети

#Суть задачи: Дана сеть (или узел и маска). Нужно найти количество доступных IP-адресов в этой сети, у которых выполняется какое-то условие (например, в двоичной записи адреса единиц больше, чем нулей).
# Используется метод net.hosts(), который автоматически исключает адрес сети и широковещательный адрес.

from ipaddress import ip_network

# Задаем сеть (можно указать адрес узла и маску, главное strict=False)
net = ip_network("124.12.192.0/255.255.248.0", strict=False)

count = 0

# net.hosts() перебирает ВСЕ допустимые узлы в этой сети
for ip in net.hosts():
    # Переводим текущий IP в полную 32-битную двоичную строку
    binary_ip = "".join(f"{int(octet):08b}" for octet in str(ip).split("."))
    
    # Условие задачи (например: единиц в двоичной записи больше, чем нулей)
    if binary_ip.count("1") > binary_ip.count("0"):
        count += 1

print("Количество подходящих узлов:", count)
# Если нужно общее количество возможных узлов без циклов:
print("Всего узлов в сети:", len(list(net.hosts())))

Количество подходящих узлов: 231
Всего узлов в сети: 2046


In [7]:

# 4. Проверка условий для широковещательного адреса (Broadcast)

# Суть задачи: Иногда в задачах спрашивают свойства не только узлов, но и самого адреса сети (network_address) или широковещательного адреса (broadcast_address).

from ipaddress import ip_network

net = ip_network("192.168.45.120/26", strict=False)

# Адрес сети (первый адрес)
print("Адрес сети:", net.network_address)

# Широковещательный адрес (самый последний адрес в сети)
print("Широковещательный адрес:", net.broadcast_address)

# Пример перевода широковещательного адреса в двоичный вид:
bin_broadcast = "".join(f"{int(octet):08b}" for octet in str(net.broadcast_address).split("."))
print("Broadcast в двоичном виде:", bin_broadcast)

Адрес сети: 192.168.45.64
Широковещательный адрес: 192.168.45.127
Broadcast в двоичном виде: 11000000101010000010110101111111
